# Desafio 1 – Gerenciamento de Estoque com DQN
Caio Couto Soares (12021BCC004) · Estevão Lorena Amorim (12511BCC044)

**Como usar:** clique em *Ambiente de execução → Executar tudo*. Ao final, os 3 gráficos aparecem aqui e o `resultados.zip` é baixado.

## 1. Bibliotecas

In [ ]:
import json
import os
import random
from collections import deque

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import Image, display

print("PyTorch", torch.__version__)

## 2. Ambiente de estoque (MDP) e políticas de referência

In [ ]:
"""Ambiente de gerenciamento de estoque (MDP) – Desafio 1.

Uma loja vende um único produto. No início de cada dia o agente decide
quantas unidades pedir ao fornecedor. O pedido chega no dia seguinte.

Estado   s = (estoque atual, pedido a caminho, dia da semana)
Ação     a ∈ {0, 5, 10, 15, 20} unidades
Recompensa do dia:
    +10 por unidade vendida
    -6  por unidade pedida
    -0,5 por unidade parada no estoque no fim do dia
    -4  por unidade que o cliente queria e não havia (venda perdida)
Episódio: 30 dias.
"""
import numpy as np


class EstoqueEnv:
    ACOES = (0, 5, 10, 15, 20)
    CAPACIDADE = 50
    HORIZONTE = 30
    PRECO_VENDA = 10.0
    CUSTO_COMPRA = 6.0
    CUSTO_ESTOQUE = 0.5
    PENALIDADE_FALTA = 4.0
    # demanda média (Poisson) de segunda (0) a domingo (6)
    DEMANDA_MEDIA = (5, 5, 6, 6, 8, 12, 10)

    def __init__(self, seed=None):
        self.rng = np.random.default_rng(seed)
        self.n_acoes = len(self.ACOES)
        self.dim_estado = 2 + 7  # estoque, a caminho, dia da semana (one-hot)
        self.reset()

    # ------------------------------------------------------------------
    def reset(self, seed=None):
        """Começa um novo episódio e devolve o estado inicial."""
        if seed is not None:
            self.rng = np.random.default_rng(seed)
        self.estoque = int(self.rng.integers(10, 31))
        self.transito = 0
        self.dia = int(self.rng.integers(0, 7))
        self.t = 0
        return self.observacao()

    def observacao(self):
        """Estado normalizado que entra na rede neural."""
        dia_onehot = np.zeros(7, dtype=np.float32)
        dia_onehot[self.dia] = 1.0
        base = np.array(
            [self.estoque / self.CAPACIDADE, self.transito / max(self.ACOES)],
            dtype=np.float32,
        )
        return np.concatenate([base, dia_onehot])

    # ------------------------------------------------------------------
    def step(self, idx_acao):
        """Executa um dia. Devolve (estado, recompensa, fim, info)."""
        pedido = self.ACOES[idx_acao]

        # 1) chega o pedido feito no dia anterior (o que passar da capacidade é perdido)
        self.estoque = min(self.estoque + self.transito, self.CAPACIDADE)

        # 2) demanda aleatória do dia
        demanda = int(self.rng.poisson(self.DEMANDA_MEDIA[self.dia]))
        vendas = min(demanda, self.estoque)
        faltas = demanda - vendas
        self.estoque -= vendas

        # 3) recompensa do dia
        recompensa = (
            self.PRECO_VENDA * vendas
            - self.CUSTO_COMPRA * pedido
            - self.CUSTO_ESTOQUE * self.estoque
            - self.PENALIDADE_FALTA * faltas
        )

        # 4) atualiza o estado
        self.transito = pedido
        self.dia = (self.dia + 1) % 7
        self.t += 1
        fim = self.t >= self.HORIZONTE

        info = {
            "pedido": pedido,
            "demanda": demanda,
            "vendas": vendas,
            "faltas": faltas,
            "estoque_fim": self.estoque,
        }
        return self.observacao(), float(recompensa), fim, info


# ----------------------------------------------------------------------
# Políticas de referência (para comparar com o DQN)
# ----------------------------------------------------------------------
def criar_politica_aleatoria(seed=0):
    rng = np.random.default_rng(seed)  # gerador próprio: não mexe na demanda do ambiente
    return lambda env, obs: int(rng.integers(0, env.n_acoes))


def criar_politica_fixa(unidades=10):
    idx = EstoqueEnv.ACOES.index(unidades)
    return lambda env, obs: idx


def criar_politica_reposicao(nivel=25):
    """Pede o suficiente para chegar perto de `nivel` (estoque + a caminho)."""
    acoes = np.array(EstoqueEnv.ACOES)

    def politica(env, obs):
        falta = max(0, nivel - (env.estoque + env.transito))
        return int(np.argmin(np.abs(acoes - falta)))

    return politica


# ----------------------------------------------------------------------
def avaliar_politica(escolher, n_episodios=300, semente_base=10_000):
    """Retorno total (lucro de 30 dias) de cada episódio.

    Todas as políticas usam as mesmas sementes, então enfrentam a mesma
    sequência de demanda: a comparação é justa.
    """
    env = EstoqueEnv()
    retornos = []
    for i in range(n_episodios):
        obs = env.reset(seed=semente_base + i)
        fim, total = False, 0.0
        while not fim:
            obs, r, fim, _ = env.step(escolher(env, obs))
            total += r
        retornos.append(total)
    return np.array(retornos)


def rodar_episodio(escolher, seed=123):
    """Registra um episódio dia a dia (para o gráfico de exemplo)."""
    env = EstoqueEnv()
    obs = env.reset(seed=seed)
    registro, fim = [], False
    while not fim:
        estoque_antes = env.estoque
        obs, r, fim, info = env.step(escolher(env, obs))
        info["recompensa"] = r
        info["estoque_inicio"] = estoque_antes
        registro.append(info)
    return registro

## 3. DQN (rede Q, replay buffer, rede alvo)

In [ ]:
"""DQN (PyTorch): rede Q, replay buffer, rede alvo e epsilon-greedy."""
import random
from collections import deque

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


class RedeQ(nn.Module):
    """Estado -> valor Q de cada ação."""

    def __init__(self, dim_estado, n_acoes, ocultas=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim_estado, ocultas),
            nn.ReLU(),
            nn.Linear(ocultas, ocultas),
            nn.ReLU(),
            nn.Linear(ocultas, n_acoes),
        )

    def forward(self, x):
        return self.net(x)


class ReplayBuffer:
    def __init__(self, capacidade):
        self.dados = deque(maxlen=capacidade)

    def add(self, s, a, r, s2):
        self.dados.append((s, a, r, s2))

    def sample(self, n):
        lote = random.sample(self.dados, n)
        s, a, r, s2 = zip(*lote)
        return (
            torch.as_tensor(np.array(s), dtype=torch.float32),
            torch.as_tensor(a, dtype=torch.int64),
            torch.as_tensor(r, dtype=torch.float32),
            torch.as_tensor(np.array(s2), dtype=torch.float32),
        )

    def __len__(self):
        return len(self.dados)


class AgenteDQN:
    def __init__(
        self,
        dim_estado,
        n_acoes,
        gamma=0.99,
        lr=1e-3,
        batch=64,
        capacidade_buffer=20_000,
        passos_atualiza_alvo=250,
        escala_recompensa=0.1,  # recompensas ~±100 -> ~±10 (treino mais estável)
    ):
        self.n_acoes = n_acoes
        self.gamma = gamma
        self.batch = batch
        self.passos_atualiza_alvo = passos_atualiza_alvo
        self.escala = escala_recompensa

        self.q = RedeQ(dim_estado, n_acoes)
        self.alvo = RedeQ(dim_estado, n_acoes)
        self.alvo.load_state_dict(self.q.state_dict())
        self.alvo.eval()

        self.otimizador = torch.optim.Adam(self.q.parameters(), lr=lr)
        self.buffer = ReplayBuffer(capacidade_buffer)
        self.passos = 0

    def agir(self, estado, epsilon):
        """Epsilon-greedy: com prob. epsilon explora, senão escolhe o maior Q."""
        if random.random() < epsilon:
            return random.randrange(self.n_acoes)
        with torch.no_grad():
            q = self.q(torch.as_tensor(estado, dtype=torch.float32).unsqueeze(0))
        return int(q.argmax(dim=1).item())

    def aprender(self):
        if len(self.buffer) < self.batch:
            return None
        s, a, r, s2 = self.buffer.sample(self.batch)

        q_atual = self.q(s).gather(1, a.unsqueeze(1)).squeeze(1)
        with torch.no_grad():
            q_prox = self.alvo(s2).max(dim=1).values
            meta = r * self.escala + self.gamma * q_prox

        perda = F.smooth_l1_loss(q_atual, meta)
        self.otimizador.zero_grad()
        perda.backward()
        nn.utils.clip_grad_norm_(self.q.parameters(), 10.0)
        self.otimizador.step()

        self.passos += 1
        if self.passos % self.passos_atualiza_alvo == 0:
            self.alvo.load_state_dict(self.q.state_dict())
        return float(perda.item())

## 4. Funções de gráfico

In [ ]:
"""Gráficos dos resultados (matplotlib)."""

import matplotlib.pyplot as plt
import numpy as np


def media_movel(x, janela=50):
    x = np.asarray(x, dtype=float)
    if len(x) < janela:
        janela = max(1, len(x))
    return np.convolve(x, np.ones(janela) / janela, mode="valid")


def curva_aprendizado(retornos, referencia, caminho):
    """Lucro por episódio durante o treino, com a política de referência."""
    plt.figure(figsize=(8, 4.5))
    plt.plot(retornos, color="#9fb3d9", alpha=0.5, label="Episódio")
    mm = media_movel(retornos, 50)
    plt.plot(np.arange(len(mm)) + 49, mm, color="#14213d", lw=2.5, label="Média móvel (50)")
    plt.axhline(referencia, color="#b44f0a", ls="--", lw=2, label="Reposição até 25 (referência)")
    plt.xlabel("Episódio de treino")
    plt.ylabel("Lucro em 30 dias")
    plt.title("Curva de aprendizado do DQN")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(caminho, dpi=150)
    plt.close()


def comparacao_politicas(resultados, caminho):
    """Barras com lucro médio de cada política (barra de erro = erro padrão)."""
    nomes = list(resultados)
    medias = [resultados[n]["media"] for n in nomes]
    erros = [resultados[n]["erro_padrao"] for n in nomes]
    cores = ["#14213d" if n == "DQN" else "#9fb3d9" for n in nomes]
    plt.figure(figsize=(8, 4.5))
    plt.bar(nomes, medias, yerr=erros, color=cores, capsize=6)
    plt.ylabel("Lucro médio em 30 dias")
    plt.title("DQN x políticas de referência")
    plt.grid(axis="y", alpha=0.3)
    plt.xticks(rotation=15)
    plt.tight_layout()
    plt.savefig(caminho, dpi=150)
    plt.close()


def episodio_exemplo(registro, caminho):
    """Um episódio dia a dia: estoque, pedidos e demanda."""
    dias = np.arange(1, len(registro) + 1)
    estoque = [d["estoque_fim"] for d in registro]
    pedidos = [d["pedido"] for d in registro]
    demanda = [d["demanda"] for d in registro]
    plt.figure(figsize=(9, 4.5))
    plt.bar(dias, pedidos, color="#f4a259", alpha=0.8, label="Pedido (unidades)")
    plt.plot(dias, estoque, color="#14213d", lw=2.5, marker="o", label="Estoque no fim do dia")
    plt.plot(dias, demanda, color="#1d7a6f", lw=1.5, ls="--", marker="x", label="Demanda do dia")
    plt.xlabel("Dia do episódio")
    plt.ylabel("Unidades")
    plt.title("Exemplo de episódio com o DQN treinado")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(caminho, dpi=150)
    plt.close()

## 5. Funções de treino

In [ ]:
def epsilon_do_episodio(ep, total, eps_ini=1.0, eps_fim=0.05, fracao=0.6):
    """Decai linearmente durante os primeiros 60% dos episódios."""
    meta = max(1, int(total * fracao))
    return max(eps_fim, eps_ini - (eps_ini - eps_fim) * ep / meta)


def treinar(env, agente, episodios, warmup=500):
    retornos = []
    for ep in range(episodios):
        eps = epsilon_do_episodio(ep, episodios)
        obs = env.reset()
        fim, total = False, 0.0
        while not fim:
            acao = agente.agir(obs, eps)
            obs2, r, fim, _ = env.step(acao)
            # O episódio acaba por limite de tempo (30 dias), não por um estado
            # final do problema: por isso o alvo continua usando Q(s') no último passo.
            agente.buffer.add(obs, acao, r, obs2)
            if len(agente.buffer) >= warmup:
                agente.aprender()
            obs = obs2
            total += r
        retornos.append(total)
        if (ep + 1) % 100 == 0:
            print(
                f"episódio {ep + 1:5d} | lucro médio (últimos 100): "
                f"{np.mean(retornos[-100:]):8.1f} | epsilon: {eps:.2f}"
            )
    return retornos

## 6. Treinar o DQN

In [ ]:
# ---- Parâmetros (pode mudar) ----
EPISODIOS = 1500   # mais episódios = treino mais longo
SEMENTE = 0        # mude para repetir o experimento com outra aleatoriedade
AVALIACAO = 300    # episódios usados na avaliação final
SAIDA = "resultados"

random.seed(SEMENTE)
np.random.seed(SEMENTE)
torch.manual_seed(SEMENTE)
os.makedirs(SAIDA, exist_ok=True)

env = EstoqueEnv(seed=SEMENTE)
agente = AgenteDQN(env.dim_estado, env.n_acoes)

print("Treinando...")
retornos = treinar(env, agente, EPISODIOS)
torch.save(agente.q.state_dict(), os.path.join(SAIDA, "modelo_dqn.pt"))

## 7. Avaliar e gerar gráficos

In [ ]:
politicas = {
    "Aleatória": criar_politica_aleatoria(),
    "Pedir 10 sempre": criar_politica_fixa(10),
    "Reposição até 25": criar_politica_reposicao(25),
    "DQN": lambda e, o: agente.agir(o, 0.0),
}
resultados = {}
print("Avaliação final:")
for nome, escolher in politicas.items():
    r = avaliar_politica(escolher, n_episodios=AVALIACAO)
    resultados[nome] = {
        "media": float(r.mean()),
        "desvio": float(r.std()),
        "erro_padrao": float(r.std() / np.sqrt(len(r))),
    }
    print(f"  {nome:18s} lucro médio = {r.mean():8.1f}  (± {r.std():.1f})")

with open(os.path.join(SAIDA, "resultados.json"), "w", encoding="utf-8") as f:
    json.dump(resultados, f, ensure_ascii=False, indent=2)

curva_aprendizado(retornos, resultados["Reposição até 25"]["media"], os.path.join(SAIDA, "curva_aprendizado.png"))
comparacao_politicas(resultados, os.path.join(SAIDA, "comparacao_politicas.png"))
registro = rodar_episodio(lambda e, o: agente.agir(o, 0.0))
episodio_exemplo(registro, os.path.join(SAIDA, "episodio_exemplo.png"))
print("Gráficos salvos na pasta", SAIDA)

## 8. Ver os gráficos

In [ ]:
for nome in ["curva_aprendizado", "comparacao_politicas", "episodio_exemplo"]:
    display(Image(os.path.join(SAIDA, nome + ".png")))

## 9. Baixar os resultados

In [ ]:
import shutil
shutil.make_archive("resultados", "zip", SAIDA)
try:
    from google.colab import files
    files.download("resultados.zip")
except ImportError:
    print("Fora do Colab: o arquivo resultados.zip está na pasta atual.")